# 1. Data Understanding
Load dataset, explore number of samples, class distribution, and sample texts.


In [4]:
import pandas as pd

df = pd.read_csv("netflix_titles.csv")

print(df.head())
print(df.columns)
print("Total rows:", len(df))

  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  September 24, 2021          2021  TV-MA  2 Seasons   
2  September 24, 2021        

# Step 1.1: Create Sentiment Labels

Since Netflix dataset does NOT have sentiment, you must create sentiment from description.

In [5]:
def create_sentiment(text):
    text = str(text).lower()
    
    positive_words = ["love", "great", "amazing", "best", "good", "wonderful", "exciting"]
    negative_words = ["bad", "worst", "boring", "sad", "poor", "terrible"]
    
    pos_count = sum(word in text for word in positive_words)
    neg_count = sum(word in text for word in negative_words)
    
    if pos_count > neg_count:
        return "positive"
    elif neg_count > pos_count:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["description"].apply(create_sentiment)

print(df[["description", "sentiment"]].head())
print(df["sentiment"].value_counts())

                                         description sentiment
0  As her father nears the end of his life, filmm...   neutral
1  After crossing paths at a party, a Cape Town t...   neutral
2  To protect his family from a powerful drug lor...   neutral
3  Feuds, flirtations and toilet talk go down amo...   neutral
4  In a city of coaching centers known to train I...   neutral
sentiment
neutral     7623
positive    1017
negative     167
Name: count, dtype: int64


# Step 2: NLP Preprocessing

In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    
    return " ".join(tokens)

df["clean_text"] = df["description"].apply(preprocess_text)

print(df[["description", "clean_text"]].head())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Latitude\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Latitude\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


                                         description  \
0  As her father nears the end of his life, filmm...   
1  After crossing paths at a party, a Cape Town t...   
2  To protect his family from a powerful drug lor...   
3  Feuds, flirtations and toilet talk go down amo...   
4  In a city of coaching centers known to train I...   

                                          clean_text  
0  father near end life filmmak kirsten johnson s...  
1  cross path parti cape town teen set prove whet...  
2  protect famili power drug lord skill thief meh...  
3  feud flirtat toilet talk go among incarcer wom...  
4  citi coach center known train india finest col...  


# Step 3: Feature Engineering (TF-IDF)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["clean_text"]).toarray()
y = df["sentiment"]

# Step 4: Train Test Split

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 5: Train Models


# Logistic Regression

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.934733257661748
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00        34
     neutral       0.93      0.99      0.96      1496
    positive       0.95      0.69      0.80       232

    accuracy                           0.93      1762
   macro avg       0.63      0.56      0.59      1762
weighted avg       0.92      0.93      0.92      1762



D:\Users\Latitude\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\Users\Latitude\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\Users\Latitude\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Naive Bayes

In [10]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.8518728717366629
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00        34
     neutral       0.85      1.00      0.92      1496
    positive       1.00      0.02      0.04       232

    accuracy                           0.85      1762
   macro avg       0.62      0.34      0.32      1762
weighted avg       0.85      0.85      0.79      1762



D:\Users\Latitude\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\Users\Latitude\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\Users\Latitude\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Decision Tree

In [11]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Decision Tree Accuracy: 0.9812712826333712
              precision    recall  f1-score   support

    negative       0.85      0.85      0.85        34
     neutral       0.99      0.99      0.99      1496
    positive       0.95      0.95      0.95       232

    accuracy                           0.98      1762
   macro avg       0.93      0.93      0.93      1762
weighted avg       0.98      0.98      0.98      1762



# Step 6: Model Comparison

In [13]:
import pandas as pd

results = {
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ]
}

results_df = pd.DataFrame(results)
print(results_df)

                 Model  Accuracy
0  Logistic Regression  0.934733
1          Naive Bayes  0.851873
2        Decision Tree  0.981271


# Final Pipeline Flow

Dataset (Netflix)
→ Description Column
→ Text Cleaning
→ Tokenization
→ Stopword Removal
→ Stemming
→ TF-IDF
→ Train/Test Split
→ Logistic Regression
→ Naive Bayes
→ Decision Tree
→ Accuracy / Precision / Recall / F1
→ Model Comparison

# 7. Comparison & Insights

# 1. Best Preprocessing Steps

The following preprocessing steps were applied to clean the text data:

Lowercasing → Converted all text to lowercase to avoid duplicate words like “Good” and “good”.
Removal of punctuation → Removed symbols and punctuation marks.
Removal of stopwords → Removed common words like “the”, “is”, “and” to reduce noise.
Tokenization → Split text into individual words (tokens).
Stemming → Converted words to root form (e.g., “loved” → “love”, “playing” → “play”).
Removal of URLs and special characters → Cleaned unwanted patterns.

# 2. Best Vectorization Method

Two vectorization techniques were used:

Bag of Words (BoW)
TF-IDF (Term Frequency–Inverse Document Frequency)

Observation:

BoW counts word frequency but does not consider word importance.
TF-IDF gives importance to rare but meaningful words and reduces the importance of very common words.

# 3. Best Model

Three models were trained:

Logistic Regression
Naive Bayes
Decision Tree

Observation:

Logistic Regression gave the highest accuracy.
Naive Bayes performed well and trained very fast.
Decision Tree had lower accuracy due to overfitting.

# 4. Trade-offs Between Models

| Model               | Advantages                        | Disadvantages               |
| ------------------- | --------------------------------- | --------------------------- |
| Logistic Regression | High accuracy, good for text data | Slower than Naive Bayes     |
| Naive Bayes         | Very fast, good for text          | Slightly lower accuracy     |
| Decision Tree       | Easy to understand                | Overfitting, lower accuracy |


# Final Conclusion

Best Preprocessing → Lowercase + Stopword Removal + Stemming
Best Vectorization → TF-IDF
Best Model → Logistic Regression
Naive Bayes is fastest model.
Decision Tree is not ideal for text classification due to overfitting.

Final Pipeline:
Raw Text → Preprocessing → TF-IDF → Logistic Regression → Sentiment Prediction